# PaleoWave — 02 Terrain Analysis
Downloads USGS 3DEP 1/3 arc-second DEM tiles for Nevada fossil clusters,
merges them, derives slope/aspect/TRI terrain metrics, and extracts values
at each PBDB occurrence point to produce the ML-ready feature table.

**Outputs:**
- `data/dem/dem_merged.tif` — merged elevation model
- `data/terrain/slope.tif`, `aspect.tif`, `ruggedness.tif`, `terrain_stack.tif`
- `data/terrain/terrain_overview.png`
- `data/features_pbdb_terrain.csv` — ML-ready feature table

> **Note:** The merged DEM is ~700M pixels. Never load it fully into memory.
> All sampling uses `rasterio.open().sample()` for efficiency.

## 1. Imports & Setup

In [ ]:
import requests
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from collections import defaultdict
import matplotlib.pyplot as plt
from pathlib import Path
import time

DEM_DIR     = Path('../data/dem')
TERRAIN_DIR = Path('../data/terrain')
PBDB_DIR    = Path('../data/pbdb')
for d in [DEM_DIR, TERRAIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Ready.')

## 2. Study Areas

In [ ]:
STUDY_AREAS = {
    'humboldt_range': {'label': 'Humboldt Range (Prida/Favret)', 'bbox': (-118.4, 39.8, -117.3, 40.8)},
    'gabbs_valley':   {'label': 'Gabbs Valley (Luning/Berlin)',  'bbox': (-118.3, 38.2, -117.3, 39.2)},
}
all_lons = [v['bbox'][0] for v in STUDY_AREAS.values()] + [v['bbox'][2] for v in STUDY_AREAS.values()]
all_lats = [v['bbox'][1] for v in STUDY_AREAS.values()] + [v['bbox'][3] for v in STUDY_AREAS.values()]
FULL_BBOX = (min(all_lons), min(all_lats), max(all_lons), max(all_lats))
for k, v in STUDY_AREAS.items():
    print(f"  {v['label']} — {v['bbox']}")
print(f'Full bbox: {FULL_BBOX}')

## 3. Query USGS TNM API

In [ ]:
def query_tnm(bbox):
    url = 'https://tnmaccess.nationalmap.gov/api/v1/products'
    params = {'datasets': 'National Elevation Dataset (NED) 1/3 arc-second',
              'bbox': f'{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}',
              'prodFormats': 'GeoTIFF', 'max': 50, 'offset': 0}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    items = resp.json().get('items', [])
    print(f'Found {len(items)} total tiles')
    return items

tiles = query_tnm(FULL_BBOX)

## 4. Deduplicate — Most Recent Tile Per Location

In [ ]:
tile_groups = defaultdict(list)
for tile in tiles:
    url = tile.get('downloadURL', '')
    if not url.endswith('.tif'): continue
    tile_groups[url.split('/')[-2]].append(tile)

best_tiles = []
for key, group in sorted(tile_groups.items()):
    best = sorted(group, key=lambda t: t.get('downloadURL',''), reverse=True)[0]
    best_tiles.append(best)
    print(f'  {key:12s} -> {best["downloadURL"].split("/")[-1]}')
print(f'\n{len(best_tiles)} unique tiles (deduplicated from {len(tiles)})')

## 5. Download DEM Tiles
~100MB per tile, skips if already downloaded.

In [ ]:
def download_tile(url, dest_path):
    if dest_path.exists():
        print(f'  Already exists: {dest_path.name} ({dest_path.stat().st_size/1e6:.1f} MB)')
        return dest_path
    print(f'  Downloading {dest_path.name}...', end=' ', flush=True)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        downloaded = 0
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk); downloaded += len(chunk)
    print(f'done ({downloaded/1e6:.1f} MB)')
    return dest_path

downloaded_tiles = []
for tile in best_tiles:
    url = tile.get('downloadURL')
    path = DEM_DIR / f'{url.split("/")[-2]}.tif'
    try: downloaded_tiles.append(download_tile(url, path))
    except Exception as e: print(f'  FAILED: {e}')
    time.sleep(0.3)
print(f'\n{len(downloaded_tiles)} / {len(best_tiles)} tiles downloaded.')

## 6. Merge Tiles

In [ ]:
if not downloaded_tiles:
    raise RuntimeError('No tiles downloaded.')
print('Merging...')
src_files = [rasterio.open(t) for t in downloaded_tiles]
merged, transform = merge(src_files)
meta = {**src_files[0].meta, 'driver':'GTiff', 'height':merged.shape[1],
        'width':merged.shape[2], 'transform':transform, 'compress':'lzw'}
merged_path = DEM_DIR / 'dem_merged.tif'
with rasterio.open(merged_path, 'w', **meta) as dst:
    dst.write(merged)
for s in src_files: s.close()
print(f'Merged DEM: {merged_path}  shape={merged.shape}  res={transform.a:.6f} deg/px')

## 7. Derive Terrain Metrics
Processed in windowed chunks — never loads full raster into memory.

In [ ]:
from rasterio.windows import Window

def compute_slope_aspect(dem, cellsize=10.0):
    dem = np.pad(dem.astype(np.float32), 1, mode='edge')
    a=dem[:-2,:-2]; b=dem[:-2,1:-1]; c=dem[:-2,2:]
    d=dem[1:-1,:-2];                  f=dem[1:-1,2:]
    g=dem[2:,:-2];  h=dem[2:,1:-1];  i=dem[2:,2:]
    dz_dx = ((c+2*f+i)-(a+2*d+g))/(8*cellsize)
    dz_dy = ((g+2*h+i)-(a+2*b+c))/(8*cellsize)
    slope  = np.degrees(np.arctan(np.sqrt(dz_dx**2+dz_dy**2)))
    aspect = np.degrees(np.arctan2(-dz_dy, dz_dx))
    aspect = np.where(aspect<0, 90.0-aspect, 360.0-aspect+90.0)
    return slope.astype(np.float32), aspect.astype(np.float32)

def compute_tri(dem):
    dem_pad = np.pad(dem.astype(np.float32), 1, mode='edge')
    center  = dem_pad[1:-1, 1:-1]
    tri = sum(np.abs(dem_pad[1+di:dem_pad.shape[0]-1+di, 1+dj:dem_pad.shape[1]-1+dj]-center)
              for di in [-1,0,1] for dj in [-1,0,1] if not(di==0 and dj==0))
    return (tri/8.0).astype(np.float32)

def save_raster(array, path, meta, nodata=-9999):
    m = {**meta, 'dtype':'float32','count':1,'nodata':nodata,'compress':'lzw'}
    with rasterio.open(path,'w',**m) as dst:
        dst.write(np.where(np.isnan(array),nodata,array).astype(np.float32),1)
    print(f'  Saved: {path.name}')

CHUNK = 500
with rasterio.open(merged_path) as src:
    h, w = src.height, src.width
    dem_meta = src.meta.copy()
    dem_transform = src.transform
    nodata = src.nodata

print(f'DEM: {w}x{h} = {w*h:,} pixels. Processing in {CHUNK}-row chunks...')

slope_path  = TERRAIN_DIR/'slope.tif'
aspect_path = TERRAIN_DIR/'aspect.tif'
tri_path    = TERRAIN_DIR/'ruggedness.tif'
out_meta    = {**dem_meta, 'dtype':'float32','count':1,'nodata':-9999,'compress':'lzw'}

with (rasterio.open(merged_path) as dem_src,
      rasterio.open(slope_path,  'w', **out_meta) as slope_dst,
      rasterio.open(aspect_path, 'w', **out_meta) as aspect_dst,
      rasterio.open(tri_path,    'w', **out_meta) as tri_dst):
    for row_start in range(0, h, CHUNK):
        row_count = min(CHUNK, h-row_start)
        win = Window(0, row_start, w, row_count)
        dem_tile = dem_src.read(1, window=win).astype(np.float32)
        if nodata is not None: dem_tile[dem_tile==nodata] = np.nan
        slope, aspect = compute_slope_aspect(dem_tile)
        tri = compute_tri(dem_tile)
        for arr, dst in [(slope,slope_dst),(aspect,aspect_dst),(tri,tri_dst)]:
            out = np.where(np.isnan(arr),-9999,arr).astype(np.float32)
            dst.write(out.reshape(1,row_count,w), window=win)
        if row_start % 5000 == 0: print(f'  ... row {row_start}/{h}')

print('Terrain metrics saved.')

## 8. Visualization
Downsampled (step=10) to avoid memory issues with the large raster.

In [ ]:
pbdb_nv = pd.read_csv(PBDB_DIR/'pbdb_occurrences_clean.csv')
pbdb_nv = pbdb_nv[pbdb_nv['in_nevada']==True]
step = 10

arrays = {}
for name, path in [('elev',merged_path),('slope',slope_path),('tri',tri_path)]:
    with rasterio.open(path) as src:
        out_h, out_w = src.height//step, src.width//step
        arrays[name] = src.read(1, out_shape=(out_h,out_w), resampling=Resampling.average)

fig, axes = plt.subplots(1,3,figsize=(18,6))
fig.patch.set_facecolor('#0d1117')
for (arr, title, cmap), ax in zip(
    [(arrays['elev'],'Elevation (m)','terrain'),
     (arrays['slope'],'Slope (deg)','YlOrRd'),
     (arrays['tri'],'Ruggedness (TRI)','magma')], axes):
    ax.set_facecolor('#0d1117')
    im = ax.imshow(arr,cmap=cmap,vmin=np.nanpercentile(arr,2),vmax=np.nanpercentile(arr,98))
    plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04).ax.yaxis.set_tick_params(color='white')
    ax.set_title(title,color='white',fontsize=11); ax.axis('off')
    px = ((pbdb_nv['longitude']-dem_transform.c)/(dem_transform.a*step)).values
    py = ((pbdb_nv['latitude'] -dem_transform.f)/(dem_transform.e*step)).values
    ax.scatter(px,py,s=40,color='#00ffcc',edgecolors='white',linewidths=0.5,zorder=5)
axes[0].scatter([],[],color='#00ffcc',edgecolors='white',s=40,label='PBDB occurrences')
axes[0].legend(facecolor='#1c1c1c',labelcolor='white',fontsize=8)
fig.suptitle('PaleoWave — Nevada Terrain Analysis',color='white',fontsize=14,y=1.01)
plt.tight_layout()
viz_path = TERRAIN_DIR/'terrain_overview.png'
plt.savefig(viz_path,dpi=150,bbox_inches='tight',facecolor='#0d1117')
plt.show()
print(f'Saved: {viz_path}')

## 9. Extract Terrain Values at PBDB Points
Uses `src.sample()` — fast vectorized read, no full raster load.

In [ ]:
pbdb_nv = pd.read_csv(PBDB_DIR/'pbdb_occurrences_clean.csv')
pbdb_nv = pbdb_nv[pbdb_nv['in_nevada']==True].copy()
coords  = list(zip(pbdb_nv['longitude'], pbdb_nv['latitude']))

for col, path in [
    ('elevation_m', merged_path),
    ('slope_deg',   slope_path),
    ('aspect_deg',  aspect_path),
    ('tri',         tri_path),
]:
    with rasterio.open(path) as src:
        pbdb_nv[col] = [v[0] for v in src.sample(coords)]
    print(f'  {col} done')

# Clean: drop nodata, fix aspect
pbdb_nv = pbdb_nv[pbdb_nv['elevation_m'] > -9999].copy()
pbdb_nv['aspect_deg'] = pbdb_nv['aspect_deg'] % 360

features_path = Path('../data/features_pbdb_terrain.csv')
pbdb_nv.to_csv(features_path, index=False)
print(f'\nFeature table saved: {features_path}')
print(f'Records: {len(pbdb_nv)}')
print(pbdb_nv[['taxon_name','elevation_m','slope_deg','aspect_deg','tri']].describe().round(2))

## Done

- `data/dem/dem_merged.tif` — merged DEM (700M pixels, ~2GB)
- `data/terrain/slope.tif`, `aspect.tif`, `ruggedness.tif`
- `data/terrain/terrain_overview.png`
- `data/features_pbdb_terrain.csv` — 27 records with terrain features

**Next:** `03_ml_model.ipynb`